# Microsoft Agent Framework

## III. Laboratorio 03 · Incorporación de conocimiento al agente: RAG y diseño de Tools
**Proyecto Integrador:** Agente Bancario, el copiloto con IA para asesores comerciales (agencia y call center) de **BancAnalytics**, un banco peruano ficticio.

Este notebook continúa el `laboratorio-02.ipynb`. Allí el agente recibía la base de conocimiento **completa dentro del system prompt** (unos 17 mil caracteres por turno), y la reflexión final dejó pendiente *"reemplazar la base de conocimiento incrustada por RAG"*. En este laboratorio el mismo agente (rol, alcance, restricciones, memoria y compactación) pasa a responder con los fragmentos que recupera de **ChromaDB**, a partir de embeddings de **`text-embedding-3-small`**.

| Sección | Contenido |
|---|---|
| 0 | Material teórico utilizado |
| 1 | Validación del Laboratorio 2: agente, memoria y compactación |
| 2 | Documentos sintéticos (Laboratorio 1: `documents/` y `outputs/`) |
| 3 | Estrategia de chunking (**pregunta 1**) |
| 4 | Estructura de los chunks y metadatos (**pregunta 2**) |
| 5 | Embeddings con Azure OpenAI mediante Agent Framework |
| 6 | Almacenamiento y recuperación en ChromaDB, con y sin filtro |
| 7 | Integración del RAG con el agente del Laboratorio 2 |
| 8 | Pruebas y validación del RAG (**pregunta 3**) |
| 9 | Diseño de Tools y evaluación de Agent as a Tool (**pregunta 4**) |
| 10 | Checklist del resultado esperado y reflexión |

**Tecnologías (las únicas permitidas por la guía):** Microsoft Agent Framework · `gpt-5.6-luna` para las respuestas · `text-embedding-3-small` para los embeddings · ChromaDB local · documentos 100 % sintéticos · autenticación con Microsoft Entra ID (`DefaultAzureCredential`), sin API keys.

### 0. Material teórico utilizado
| Paso del laboratorio | Material de teoría | Qué se reutiliza y qué cambia |
|---|---|---|
| Chunking (sección 3) | `clase-04-rag/chunking.ipynb`: *3.4 Structure-based chunking* (`structure_based_chunking`), *"Se pueden combinar diferentes enfoques"* (contexto añadido a cada chunk) y *Consideraciones al particionar* | Se divide por los **encabezados Markdown** que generó Document Intelligence en lugar de por las claves de un JSON. El contexto se añade con el título del documento y de la sección, **sin generarlo con un LLM** para no inventar contenido. |
| Estructura y metadatos (sección 4) | `chunking.ipynb` (conservar el origen de cada fragmento) y `embeddings.ipynb` (`metadatas` en la colección) | Se adopta el formato `id`, `text`, `metadata` de la guía, solo con valores simples. |
| Embeddings (sección 5) | `clase-04-rag/embeddings.ipynb`: *¿Cómo se utilizan en RAG?* y *Métricas de similitud* | `SentenceTransformer` se reemplaza por `OpenAIEmbeddingClient` (`text-embedding-3-small`) con `DefaultAzureCredential`. |
| ChromaDB (sección 6) | `embeddings.ipynb`: *Embeddings con base de datos* (`PersistentClient`, `get_or_create_collection`, `query`) | Se usa `embedding_function=None` y la métrica `cosine`, y los vectores se envían explícitamente. |
| Filtros (secciones 6 y 8) | `clase-04-rag/rag.ipynb`: *RAG mediante filtros de metadatos* y *RAG mediante metadatos y contenido* (híbrido) | El filtro por metadatos se hace con `where={"categoria": ...}` de Chroma sobre la búsqueda semántica. |
| Integración con el agente (sección 7) | `rag.ipynb`: *5. Ejemplo conceptual con Agent Framework* (construir la consulta con el contexto recuperado) | El contexto se envía al **agente del Laboratorio 2**, con su sesión, memoria y compactación (`clase-03-prompt-memoria/compaction.ipynb`). |
| Pruebas (sección 8) | `chunking.ipynb`: *Reranking* y las limitaciones de la similitud; `embeddings.ipynb`: *Perspectivas de RAG* (Self-RAG / Corrective RAG) | Se valida que una alta similitud no garantiza que el fragmento contenga la respuesta. |
| Tools y Agent as a Tool (sección 9) | `clase-05-tool-mcp-a2a/tools.ipynb` (*Tools vs. RAG*, `@tool`), `agent-tool.ipynb` (*¿Cuándo utilizarlo?*, *Agent as a Tool vs. Workflow*) y `mcp.ipynb` (*MCP vs. Tools tradicionales*) | Solo se diseña, sin implementar. |

### 1. Validación del Laboratorio 2
Se reconstruye el agente del Laboratorio 2 con el **mismo código**: plantilla de 5 componentes, `datos`, `InMemoryHistoryProvider` y `CompactionProvider(after_strategy=SummarizationStrategy)`. Luego se comprueba que la memoria y la compactación siguen funcionando antes de agregar el RAG.

In [1]:
import os
import re
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from agent_framework import Agent, InMemoryHistoryProvider, SummarizationStrategy, CompactionProvider
from agent_framework.openai import OpenAIChatClient, OpenAIEmbeddingClient

pd.set_option("display.max_colwidth", 120)

# Variables de entorno (mismo .env del Laboratorio 2 + modelo de embeddings)
load_dotenv()
endpoint_llm = os.environ["FOUNDRY_MODEL_ENDPOINT"]
modelo_llm = os.environ["FOUNDRY_MODEL_DEPLOYMENT"]
modelo_embedding = os.environ["FOUNDRY_EMBEDDING_DEPLOYMENT"]

# Identidad asignada (Microsoft Entra ID): sin API keys en el código ni en el .env
credential = DefaultAzureCredential()

client = OpenAIChatClient(
    model=modelo_llm,  # gpt-5.6-luna
    base_url=endpoint_llm,
    credential=credential,
)
print("LLM:", modelo_llm, "| Embeddings:", modelo_embedding)

LLM: gpt-5.6-luna | Embeddings: text-embedding-3-small


In [2]:
# -----------------------------------------------------------------------------
# Código del Laboratorio 2 (sin cambios): plantilla, base de conocimiento y datos del prompt
plantilla_prompt = """
ROL:
Actúa como {rol}.

OBJETIVO:
{objetivo}

CONTEXTO:
{contexto}

RESTRICCIONES:
{restricciones}

FORMATO DE SALIDA:
{formato_salida}
"""

OUT_DIR = Path("outputs")
kb = json.loads((OUT_DIR / "bancanalytics_kb_index.json").read_text(encoding="utf-8"))

def formatear_kb(registros):
    bloques = []
    for d in registros:
        paginas = ",".join(str(p) for p in d.get("paginas", []))
        bloques.append(f"[{d['id']} | {d['fuente']} | pág. {paginas}]\n{d['contenido']}")
    return "\n\n".join(bloques)

kb_texto = formatear_kb(kb)

RESTRICCIONES_LAB2 = (
    "- REGLA CLAVE: nunca presentes una oferta pre-aprobada como aprobada. Usa "
    "'oferta pre-aprobada de hasta S/ X, sujeta a validación' y recuerda su vigencia.\n"
    "- Usa solo datos de la base de conocimiento o del perfil compartido por el "
    "asesor. Si un dato no está, dilo y no lo inventes.\n"
    "- No apruebes, garantices ni prometas créditos o desembolsos; no ejecutes operaciones.\n"
    "- Recomienda variantes según el ingreso mínimo y la necesidad del cliente; evita "
    "el sobreendeudamiento (cuotas totales <= 40% del ingreso neto, referencial) y la venta agresiva.\n"
    "- Las simulaciones de cuota son referenciales (sistema francés con TEA convertida a "
    "tasa mensual); muestra los supuestos.\n"
    "- Nunca solicites ni repitas claves, PIN, token, códigos SMS, CVV ni números "
    "completos de tarjeta o documento; enmascáralos.\n"
    "- ALCANCE. Antes de responder, clasifica internamente la consulta (no muestres el razonamiento):\n"
    "  * EN_ALCANCE: Tarjeta de Crédito, Crédito Efectivo, información institucional "
    "de BancAnalytics (canales, horarios, reclamos, seguridad) o preguntas sobre qué puedes hacer.\n"
    "  * OTRA_ESPECIALIDAD: temas bancarios fuera de tus productos (crédito hipotecario, "
    "crédito vehicular, cuentas de ahorro, depósitos a plazo, CTS, seguros, inversiones, "
    "banca empresas). No respondas el fondo: indica que corresponde a otro agente "
    "especializado, nómbralo y ofrece ayuda en tus temas.\n"
    "  * FUERA_DE_ALCANCE: temas no bancarios (deportes, recetas, programación, etc.). "
    "Recházalo con amabilidad y enumera los temas en los que sí puedes ayudar."
)

def formato_salida(ejemplo_fuente):
    return (
        "La PRIMERA línea siempre es exactamente: 'ALCANCE: EN_ALCANCE', "
        "'ALCANCE: OTRA_ESPECIALIDAD' o 'ALCANCE: FUERA_DE_ALCANCE'.\n"
        "Si es EN_ALCANCE, responde en español, dirigido al asesor, con estas secciones breves:\n"
        "1) Resumen de la consulta; 2) Respuesta para el asesor; "
        "3) Mensaje sugerido para el cliente (listo para usar); "
        "4) Oportunidades de oferta (o 'Ninguna'); "
        f"5) Fuentes: ids y documentos usados (ej. {ejemplo_fuente}).\n"
        "Si es OTRA_ESPECIALIDAD o FUERA_DE_ALCANCE, responde en máximo 3 líneas después de la etiqueta.\n"
        "EJEMPLOS:\n"
        "Asesor: ¿Qué ingreso mínimo pide la tarjeta Platinum?\n"
        "Copiloto: ALCANCE: EN_ALCANCE\n1) Resumen: ingreso mínimo de la Platinum. "
        f"2) Respuesta: S/ 6,000 mensuales... 5) Fuentes: {ejemplo_fuente}\n"
        "Asesor: El cliente quiere un préstamo para comprar su departamento.\n"
        "Copiloto: ALCANCE: OTRA_ESPECIALIDAD\nEl crédito hipotecario lo atiende el agente "
        "de Créditos Hipotecarios. Puedo ayudarte con Tarjeta de Crédito, Crédito Efectivo "
        "e información del banco.\n"
        "Asesor: ¿Quién ganó el clásico de fútbol ayer?\n"
        "Copiloto: ALCANCE: FUERA_DE_ALCANCE\nNo puedo ayudar con ese tema. Puedo apoyarte "
        "con Tarjeta de Crédito, Crédito Efectivo, ofertas del cliente, simulaciones de "
        "cuota y canales o reclamos de BancAnalytics."
    )

datos = {
    "rol": (
        "copiloto con IA de los asesores comerciales (agencia y call center) de "
        "BancAnalytics, un banco peruano. Eres especialista en Tarjeta de Crédito "
        "(variantes Clásica, Oro, Platinum y Black), Crédito Efectivo e información "
        "institucional del banco (canales, horarios, reclamos y seguridad)."
    ),
    "objetivo": (
        "Ayudar al asesor a atender y a vender de forma responsable: "
        "(1) responder dudas sobre el banco y sus productos citando la fuente; "
        "(2) interpretar el perfil y las ofertas vigentes del cliente "
        "(pre-aprobadas o aprobadas); "
        "(3) recomendar qué producto o variante ofrecer y con qué argumentos; "
        "(4) simular de forma referencial la cuota del Crédito Efectivo; y "
        "(5) resumir la atención y redactar mensajes para el cliente. "
        "El asesor siempre toma la decisión final; tú no ejecutas operaciones."
    ),
    "contexto": (
        "Quien te escribe es el ASESOR, no el cliente. País: Perú; regulación: SBS, "
        "Indecopi y Libro de Reclamaciones; moneda: soles (S/). "
        "El asesor puede compartirte el perfil del cliente (ingresos, productos, "
        "ofertas vigentes y necesidad); úsalo durante toda la conversación.\n"
        "Tu única fuente de información de productos es la siguiente base de "
        "conocimiento (documentos sintéticos de BancAnalytics):\n"
        "### BASE DE CONOCIMIENTO ###\n"
        f"{kb_texto}\n"
        "### FIN DE LA BASE DE CONOCIMIENTO ###"
    ),
    "restricciones": RESTRICCIONES_LAB2,
    "formato_salida": formato_salida("[bca-tc-03 | 02_BancAnalytics_Tarjeta_de_Credito.pdf]"),
}
system_prompt_lab2 = plantilla_prompt.format(**datos)
print(f"System prompt del Laboratorio 2: {len(system_prompt_lab2):,} caracteres (con {len(kb)} secciones incrustadas)")

System prompt del Laboratorio 2: 17,412 caracteres (con 22 secciones incrustadas)


In [3]:
# -----------------------------------------------------------------------------
# Memoria y compactación del Laboratorio 2 (configuración final adoptada)
PROMPT_RESUMEN = """
Resume los mensajes antiguos de una atención de un asesor de BancAnalytics.
Conserva SIEMPRE, de forma literal:
- Perfil del cliente: nombre (o iniciales), ingreso, situación laboral, productos actuales.
- Ofertas vigentes con su TIPO (pre-aprobada o aprobada), monto, tasa y vigencia.
- Necesidad del cliente, decisiones tomadas, simulaciones realizadas y pendientes.
Nunca conviertas una oferta pre-aprobada en aprobada. No inventes información.
No incluyas datos sensibles (DNI completo, claves, números de tarjeta).
No copies fragmentos del CONTEXTO RECUPERADO: se vuelven a buscar en cada pregunta.
"""

def nueva_estrategia(target_count, threshold):
    return SummarizationStrategy(
        client=OpenAIChatClient(model=modelo_llm, base_url=endpoint_llm, credential=credential),
        target_count=target_count,  # mensajes recientes que se conservan enteros
        threshold=threshold,        # margen adicional antes de disparar la compactación
        prompt=PROMPT_RESUMEN,
    )

def crear_agente_compactado(instrucciones, target_count, threshold):
    historial = InMemoryHistoryProvider("historial", load_messages=True, skip_excluded=True)
    compactacion = CompactionProvider(
        after_strategy=nueva_estrategia(target_count, threshold),
        history_source_id=historial.source_id,
    )
    return Agent(
        client=client,
        name="CopilotoAsesorBancAnalytics",
        instructions=instrucciones,
        default_options={"store": False},  # el historial lo gestiona el framework
        context_providers=[historial, compactacion],
    )

def mensajes_de(sesion, fuente="historial"):
    return sesion.state.get(fuente, {}).get("messages", [])

def estado_historial(sesion):
    hist = mensajes_de(sesion)
    excluidos = sum(1 for m in hist if (m.additional_properties or {}).get("_excluded"))
    resumenes = sum(1 for m in hist if "_summary_of_message_ids" in (((m.additional_properties or {}).get("_group")) or {}))
    return {"almacenados": len(hist), "enviados_al_modelo": len(hist) - excluidos,
            "excluidos": excluidos, "resumenes": resumenes}

TURNOS_ENTEROS = 6  # valor definido en la pregunta 2 del Laboratorio 2

In [4]:
# Validación rápida con valores pequeños (target_count=4, threshold=2), igual que la sección 4.2 del Laboratorio 2
agente_lab2 = crear_agente_compactado(system_prompt_lab2, target_count=4, threshold=2)
sesion_lab2 = agente_lab2.create_session()

turnos_validacion = [
    "Atiendo al cliente Luis M.: ingreso neto de S/ 7,200, dependiente, tiene un Crédito "
    "Efectivo APROBADO de S/ 35,000 a TEA 18.9% que vence en 10 días, y quiere remodelar su casa.",
    "¿Cuál es el horario de atención de las agencias?",
    "¿Cuál es el número de Banca Telefónica?",
    "¿Los Puntos Analytics vencen?",
]
for i, msg in enumerate(turnos_validacion, start=1):
    await agente_lab2.run(msg, session=sesion_lab2)
    print(f"Turno {i}: {estado_historial(sesion_lab2)}")

r = await agente_lab2.run(
    "Recuérdame el monto, tipo de oferta, TEA y vigencia del Crédito Efectivo de mi cliente, y su ingreso.",
    session=sesion_lab2,
)
print(f"Turno 5: {estado_historial(sesion_lab2)}\n")
print(r.text[:900])

Turno 1: {'almacenados': 2, 'enviados_al_modelo': 2, 'excluidos': 0, 'resumenes': 0}


Turno 2: {'almacenados': 4, 'enviados_al_modelo': 4, 'excluidos': 0, 'resumenes': 0}


Turno 3: {'almacenados': 6, 'enviados_al_modelo': 6, 'excluidos': 0, 'resumenes': 0}


Turno 4: {'almacenados': 8, 'enviados_al_modelo': 8, 'excluidos': 0, 'resumenes': 0}


Turno 5: {'almacenados': 11, 'enviados_al_modelo': 7, 'excluidos': 4, 'resumenes': 1}

ALCANCE: EN_ALCANCE

1) **Resumen de la consulta**  
Datos principales del Crédito Efectivo y del ingreso de Luis M.

2) **Respuesta para el asesor**  
- **Monto:** S/ 35,000  
- **Tipo de oferta:** Crédito Efectivo **aprobado**  
- **TEA:** 18.9%  
- **Vigencia:** vence en **10 días**, según el perfil compartido  
- **Ingreso neto:** S/ 7,200 mensuales  
- **Situación laboral:** dependiente  

3) **Mensaje sugerido para el cliente**  
“Luis, tienes un Crédito Efectivo **aprobado por S/ 35,000**, con una TEA de 18.9%, vigente por 10 días. Tu ingreso neto registrado es de S/ 7,200 mensuales. Antes de aceptar, revisaremos la cuota, el seguro, la TCEA y las demás condiciones de la cartilla.”

4) **Oportunidades de oferta**  
Priorizar la aceptación del Crédito Efectivo aprobado dentro del plazo de vigencia. No presentarlo como una oferta pre-aprobada.

5) **Fuentes**  
- Datos del perfil co


**Observaciones de la validación del Laboratorio 2:**
- El agente del Laboratorio 2 se reconstruyó con el mismo código y funciona con `gpt-5.6-luna` y `DefaultAzureCredential`. Su system prompt mide **17,412 caracteres**, porque incluye las 22 secciones del Lab 1.
- **Memoria:** el historial de la sesión crece de 2 en 2 mensajes (usuario y asistente) durante los turnos 1 a 4.
- **Compactación:** con los valores de prueba (`target_count=4`, `threshold=2`), al cerrar el turno 5 la sesión guarda 11 mensajes, pero **solo 7 se envían al modelo**: 4 quedan excluidos y se reemplazan por **1 resumen**.
- **El dato clave sobrevivió:** el turno 1 ya no se envía al modelo y, aun así, el agente respondió monto (S/ 35,000), tipo (**aprobado**), TEA (18.9 %), vigencia (10 días) e ingreso (S/ 7,200).
- ✅ **Checkpoint:** system prompt con alcance y restricciones, sesión con memoria de corto plazo y compactación probada. Este mismo agente es el que recibe el RAG en la sección 7.

### 2. Documentos sintéticos del proyecto
Se reutiliza lo que ya está en el repositorio. **No se vuelven a procesar los PDF con Document Intelligence.**

| Archivo | Origen | Uso en el Laboratorio 3 |
|---|---|---|
| `documents/01_BancAnalytics_Informacion_Institucional.pdf` | PDF sintético (generado con IA, Lab 1) | Fuente citada y propiedades del PDF (título, autor, páginas, fecha) |
| `documents/02_BancAnalytics_Tarjeta_de_Credito.pdf` | PDF sintético | Ídem |
| `documents/03_BancAnalytics_Credito_Efectivo.pdf` | PDF sintético | Ídem |
| `outputs/bancanalytics_layout_*.md` | Salida de `prebuilt-layout` (Lab 1), en Markdown con encabezados y tablas | **Texto que se fragmenta** |
| `outputs/bancanalytics_kb_index.json` | 22 secciones del Lab 1 (Paso 6), con metadatos | Metadatos de documento: `categoria`, `tipo_documento`, `fecha_publicacion`, `vigente`, `origen` |
| `documents/04_BancAnalytics_Factura_Proveedor.pdf` y `outputs/factura_prebuilt_invoice_raw.json` | Factura sintética de un proveedor de plásticos (Lab 1, `prebuilt-invoice`) | **Se excluye del RAG** (ver la justificación debajo) |

**¿Por qué se excluye la factura?** Es un documento operativo de compras (BancAnalytics le compra plásticos a un proveedor), no información que el asesor necesite para atender a un cliente. Indexarla mezclaría importes de factura (S/, IGV, cantidades) con tasas y montos de productos y podría contaminar la recuperación en preguntas como *"¿cuánto cuesta la tarjeta?"*. Quedan **3 documentos y 3 categorías**, por encima del mínimo de 2 que pide la guía.

In [5]:
from pypdf import PdfReader

DOCS_DIR = Path("documents")

DOCUMENTOS_FUENTE = [
    # id del documento, Markdown de Document Intelligence (Lab 1), PDF original
    {"id": "bca-inst", "layout": "bancanalytics_layout_institucional.md",
     "pdf": "01_BancAnalytics_Informacion_Institucional.pdf"},
    {"id": "bca-tc", "layout": "bancanalytics_layout_tarjeta.md",
     "pdf": "02_BancAnalytics_Tarjeta_de_Credito.pdf"},
    {"id": "bca-ce", "layout": "bancanalytics_layout_credito_efectivo.md",
     "pdf": "03_BancAnalytics_Credito_Efectivo.pdf"},
]

def propiedades_pdf(ruta):
    lector = PdfReader(ruta)
    meta = lector.metadata or {}
    return {
        "pdf_title": meta.get("/Title"),
        "pdf_author": meta.get("/Author"),
        "pdf_creator": meta.get("/Creator"),
        "pdf_producer": meta.get("/Producer"),
        "pdf_keywords": meta.get("/Keywords"),
        "pdf_creation_date": str(meta.get("/CreationDate")),
        "pdf_paginas": len(lector.pages),
    }

# Metadatos de documento que ya se calcularon en el Lab 1 (una fila por PDF)
meta_lab1 = {}
for registro in kb:
    meta_lab1.setdefault(registro["fuente"], registro)

documentos = []
for d in DOCUMENTOS_FUENTE:
    lab1 = meta_lab1[d["pdf"]]
    documentos.append({
        "id": d["id"],
        "titulo": None,  # se toma del encabezado H1 del Markdown en la sección 3
        "contenido": (OUT_DIR / d["layout"]).read_text(encoding="utf-8"),
        "categoria": lab1["categoria"],
        "tipo_documento": lab1["tipo_documento"],
        "fuente": d["pdf"],
        "fecha_publicacion": lab1["fecha_publicacion"][:10],
        "vigente": lab1["vigente"],
        "origen": lab1["origen"],
        **propiedades_pdf(DOCS_DIR / d["pdf"]),
    })

pd.DataFrame(documentos).drop(columns=["contenido", "titulo"])

,id,categoria,tipo_documento,fuente,fecha_publicacion,vigente,origen,pdf_title,pdf_author,pdf_creator,pdf_producer,pdf_keywords,pdf_creation_date,pdf_paginas
0,bca-inst,Institucional,Información institucional,01_BancAnalytics_Informacion_Institucional.pdf,2026-11-14,True,sintético (generado con IA para el laboratorio),Información institucional,BancAnalytics,(unspecified),ReportLab PDF Library - (opensource),,D:20261114101302-05'00',2
1,bca-tc,Tarjeta de Crédito,Ficha de producto,02_BancAnalytics_Tarjeta_de_Credito.pdf,2026-11-14,True,sintético (generado con IA para el laboratorio),Tarjeta de Crédito,BancAnalytics,(unspecified),ReportLab PDF Library - (opensource),,D:20261114101302-05'00',2
2,bca-ce,Crédito Efectivo,Ficha de producto,03_BancAnalytics_Credito_Efectivo.pdf,2026-11-14,True,sintético (generado con IA para el laboratorio),Crédito Efectivo,BancAnalytics,(unspecified),ReportLab PDF Library - (opensource),,D:20261114101302-05'00',2


### 3. Estrategia de chunking
**Estrategia: división por estructura (encabezados Markdown) más contexto añadido**, basada en `structure_based_chunking` de `chunking.ipynb` (3.4) y en el ejemplo *"Se pueden combinar diferentes enfoques"*.

| Parámetro | Valor | Motivo |
|---|---|---|
| Separador | Encabezados Markdown `##`, `###` y `####` que generó `prebuilt-layout` | Cada sección de la ficha es una sola idea: variantes, costos, requisitos, preguntas frecuentes, etc. |
| Título del documento (`#`) | No genera un chunk. Su subtítulo (*"Ficha de producto para asesores · Versión 1.0 · Noviembre 2026"*) pasa a ser el metadato `version_documento` | Es información de procedencia, no contenido para responder |
| Preguntas frecuentes | Las preguntas formateadas como encabezado (`#### ¿BancAnalytics es un banco seguro?`) **no** cortan la sección | Así no se generan chunks de una sola línea |
| Tablas HTML | Se convierten a filas `Concepto → Columna: valor` y **nunca se cortan** | Un dato de tabla (por ejemplo, el ingreso mínimo de la Platinum) solo se entiende con el nombre de su fila y su columna |
| Saltos de página | Se eliminan los comentarios `PageHeader`, `PageFooter`, `PageNumber` y `PageBreak`, pero se registra la página | La FAQ que continúa en la página 2 queda en un solo chunk, con `paginas="1,2"` |
| Tamaño máximo | `MAX_CHARS = 1200` (≈ 300 tokens). Si una sección lo supera, se divide por filas o párrafos y se repite el encabezado | Es una red de seguridad. Con estos documentos la sección más larga mide 1,166 caracteres, así que no se activa |
| Solapamiento | No se usa | Las secciones son autocontenidas; el solapamiento duplicaría filas de tabla sin agregar contexto |
| Contexto añadido | Prefijo `"{título del documento} · {sección}"` en el texto de cada chunk | "Requisitos" y "Preguntas frecuentes" existen en Tarjeta **y** en Crédito Efectivo. Sin el prefijo, el fragmento no dice de qué producto habla. Se usa texto del propio documento, sin generarlo |

In [6]:
from html import unescape

MAX_CHARS = 1200

def tabla_html_a_texto(html):
    """Convierte una tabla HTML de Document Intelligence en filas 'Concepto → Columna: valor'."""
    filas = re.findall(r"<tr>(.*?)</tr>", html, flags=re.S)
    celdas = [[unescape(re.sub(r"\s+", " ", c)).strip()
               for c in re.findall(r"<t[hd]>(.*?)</t[hd]>", f, flags=re.S)] for f in filas]
    if not celdas:
        return ""
    encabezado, *cuerpo = celdas
    lineas = []
    for fila in cuerpo:
        concepto, valores = fila[0], fila[1:]
        if len(encabezado) == 2:
            lineas.append(f"- {concepto}: {valores[0]}")
        else:
            pares = " | ".join(f"{encabezado[i + 1]}: {v}" for i, v in enumerate(valores))
            lineas.append(f"- {concepto} → {pares}")
    return "\n".join(lineas)

def limpiar_markdown(md):
    md = md.replace("<!-- PageBreak -->", "\n[[SALTO_PAGINA]]\n")
    md = re.sub(r"<!--.*?-->", "", md, flags=re.S)  # PageHeader, PageFooter, PageNumber
    return re.sub(r"<table>.*?</table>", lambda m: tabla_html_a_texto(m.group(0)), md, flags=re.S)

def dividir_por_estructura(md):
    """Devuelve (título del documento, versión, [(sección, texto, páginas)])."""
    titulo_doc, version, secciones = None, None, []
    actual, buffer, pagina, paginas = None, [], 1, set()

    def cerrar():
        texto = re.sub(r"\n{3,}", "\n\n", "\n".join(buffer)).strip()
        if actual and texto:
            secciones.append((actual, texto, sorted(paginas)))

    for linea in limpiar_markdown(md).splitlines():
        if linea.strip() == "[[SALTO_PAGINA]]":
            pagina += 1
            continue
        m = re.match(r"^(#{1,4})\s+(.*)", linea)
        es_pregunta_faq = bool(m) and actual is not None and actual.startswith("Preguntas frecuentes")
        if m and not es_pregunta_faq:
            if len(m.group(1)) == 1:            # título del documento
                titulo_doc = m.group(2).strip()
                continue
            cerrar()                             # nueva sección
            actual, buffer, paginas = m.group(2).strip(), [], {pagina}
            continue
        if actual is None:                       # subtítulo bajo el H1 -> versión del documento
            if linea.strip() and version is None:
                version = linea.strip()
            continue
        buffer.append(m.group(2) if m else linea)
        if linea.strip():
            paginas.add(pagina)
    cerrar()
    return titulo_doc, version, secciones

def subdividir(texto, max_chars=MAX_CHARS):
    """Red de seguridad: corta por filas o párrafos completos, nunca a mitad de una fila."""
    if len(texto) <= max_chars:
        return [texto]
    bloques = [b for b in re.split(r"\n(?=- )|\n\n", texto) if b.strip()]
    partes, parte = [], ""
    for b in bloques:
        if parte and len(parte) + len(b) + 1 > max_chars:
            partes.append(parte.strip())
            parte = ""
        parte += b + "\n"
    if parte.strip():
        partes.append(parte.strip())
    return partes

In [7]:
chunks = []
for doc in documentos:
    titulo_doc, version, secciones = dividir_por_estructura(doc["contenido"])
    doc["titulo"] = titulo_doc
    numero = 0
    for seccion, texto, paginas in secciones:
        partes = subdividir(texto)
        for k, parte in enumerate(partes, start=1):
            numero += 1
            nombre_seccion = seccion if len(partes) == 1 else f"{seccion} (parte {k}/{len(partes)})"
            chunks.append({
                "id": f"{doc['id']}_chunk_{numero:02d}",
                # Contexto añadido: documento y sección (texto del propio documento, no generado)
                "text": f"{titulo_doc} · {nombre_seccion}\n\n{parte}",
                "metadata": {
                    "fuente": doc["fuente"],
                    "titulo": titulo_doc,
                    "seccion": nombre_seccion,
                    "categoria": doc["categoria"],
                    "tipo_documento": doc["tipo_documento"],
                    "numero_chunk": numero,
                    "paginas": ",".join(map(str, paginas)),
                    "version_documento": version,
                    "fecha_publicacion": doc["fecha_publicacion"],
                    "vigente": doc["vigente"],
                    "origen": doc["origen"],
                },
            })

# Comprobación que pide la guía: cada chunk conserva su documento de origen y tiene un id único
assert len({c["id"] for c in chunks}) == len(chunks), "ids duplicados"
assert all(c["metadata"]["fuente"] and c["metadata"]["titulo"] for c in chunks)

print(f"Documentos: {len(documentos)} | Chunks: {len(chunks)} | "
      f"longitud de los textos: {min(len(c['text']) for c in chunks)} a {max(len(c['text']) for c in chunks)} caracteres")
pd.DataFrame([{"id": c["id"], "categoria": c["metadata"]["categoria"], "seccion": c["metadata"]["seccion"],
               "paginas": c["metadata"]["paginas"], "caracteres": len(c["text"])} for c in chunks])

Documentos: 3 | Chunks: 20 | longitud de los textos: 228 a 1224 caracteres


,id,categoria,seccion,paginas,caracteres
0,bca-inst_chunk_01,Institucional,¿Quiénes somos?,1,420
1,bca-inst_chunk_02,Institucional,Datos clave,1,537
2,bca-inst_chunk_03,Institucional,Canales de atención,1,759
3,bca-inst_chunk_04,Institucional,Productos para personas,1,347
4,bca-inst_chunk_05,Institucional,Reclamos y defensa del cliente,1,466
5,bca-inst_chunk_06,Institucional,Seguridad: lo que el banco nunca hará,1,329
6,bca-inst_chunk_07,Institucional,Preguntas frecuentes de clientes,"1,2",556
7,bca-tc_chunk_01,Tarjeta de Crédito,Descripción,1,340
8,bca-tc_chunk_02,Tarjeta de Crédito,Variantes de plástico,1,1224
9,bca-tc_chunk_03,Tarjeta de Crédito,Costos y condiciones generales,1,536


In [8]:
# Tres chunks de ejemplo: una tabla, una sección que existe en dos documentos y una FAQ que cruza de página
for id_chunk in ["bca-tc_chunk_02", "bca-ce_chunk_05", "bca-inst_chunk_07"]:
    c = next(c for c in chunks if c["id"] == id_chunk)
    print("=" * 100)
    print(f"{c['id']} | {c['metadata']['fuente']} | págs. {c['metadata']['paginas']}")
    print("-" * 100)
    print(c["text"])

bca-tc_chunk_02 | 02_BancAnalytics_Tarjeta_de_Credito.pdf | págs. 1
----------------------------------------------------------------------------------------------------
Tarjeta de Crédito BancAnalytics · Variantes de plástico

- Ingreso mínimo mensual → Clásica: S/ 1,200 | Oro: S/ 3,000 | Platinum: S/ 6,000 | Black: S/ 15,000
- Línea de crédito → Clásica: S/ 500 -6,000 | Oro: S/ 3,000 - 15,000 | Platinum: S/ 10,000 - 40,000 | Black: Desde S/ 40,000
- Membresía anual → Clásica: S/ 0 | Oro: S/ 120 (exonerable) | Platinum: S/ 240 (exonerable) | Black: S/ 600 (exonerable)
- TEA compras (ref.) → Clásica: 79.9% | Oro: 69.9% | Platinum: 54.9% | Black: 39.9%
- Puntos por cada S/ 1 → Clásica: 1 pto. x S/ 5 | Oro: 1 pto. x S/ 3 | Platinum: 1 pto. x S/ 2 | Black: 1 pto. x S/ 1
- Cashback → Clásica: — | Oro: 1% en supermercados | Platinum: 2% supermercados y grifos | Black: 3% en restaurantes y viajes
- Salas VIP aeropuerto → Clásica: — | Oro: — | Platinum: 4 accesos/año | Black: Ilimitado + 1 inv

**Observaciones del chunking:**
- Se generaron **20 chunks** de 3 documentos: 7 de Institucional, 6 de Tarjeta de Crédito y 7 de Crédito Efectivo. Miden entre 228 y 1,224 caracteres. El tope de 1,200 caracteres (que se aplica al contenido, sin el prefijo) no tuvo que dividir ninguna sección, así que **ninguna tabla quedó cortada**.
- **Las tablas se entienden por separado:** en `bca-tc_chunk_02`, cada fila conserva su concepto y su columna (*"Ingreso mínimo mensual → … Platinum: S/ 6,000"*), algo que el HTML original solo daba por posición.
- **Ambigüedad resuelta con contexto:** `bca-ce_chunk_05` empieza con *"Crédito Efectivo BancAnalytics · Requisitos"*. Sin ese prefijo, no se distinguiría de los requisitos de la tarjeta (`bca-tc_chunk_04`).
- **La FAQ que cruza de página** (`bca-inst_chunk_07`, págs. 1 y 2) quedó en un solo chunk. Su primera pregunta venía como encabezado `####` y no generó un fragmento de una línea.
- El chunk *¿Quiénes somos?* se conservó. Es un encabezado con signo de interrogación, pero no es una pregunta frecuente.
- **Calidad del origen:** algunos textos conservan errores de extracción del Lab 1. Por ejemplo, *"consumos de / 1,500(Or)"* en la exoneración de membresía. No se corrigen a mano para no alterar la fuente; se registran como un punto a mejorar en el documento original.

#### Pregunta 1 · ¿Qué estrategia de chunking eligieron y por qué es adecuada para los documentos de su proyecto?

**División por estructura (encabezados Markdown de Document Intelligence), con un prefijo de contexto `"{documento} · {sección}"` y un tope de seguridad de 1,200 caracteres sin solapamiento.**

Es adecuada porque:
1. **Los documentos ya vienen estructurados.** Son fichas para asesores con una sección por tema (variantes, costos, requisitos, tipos de oferta, FAQ), y `prebuilt-layout` las devolvió como encabezados `##`, `###` y `####`. Cada sección responde un tipo de pregunta del asesor, así que *una sección = un chunk* produce fragmentos coherentes y completos.
2. **Los datos críticos están en tablas.** Un chunking de **tamaño fijo** o con **solapamiento** cortaría la tabla de variantes o la de tipos de oferta a mitad de fila. Entonces se perdería a qué variante corresponde un ingreso mínimo, o la nota *"nunca presentar una oferta pre-aprobada como aprobada"*, que va justo después de la tabla. Al dividir por estructura, cada tabla queda entera.
3. **El chunking semántico no aporta aquí:** las fronteras temáticas ya las marcan los encabezados. Además, requiere embeddings por oración y un umbral adicional, lo que agrega costo sin un beneficio visible en documentos tan cortos.
4. **El prefijo de contexto** es la técnica de *"combinar enfoques"* de `chunking.ipynb`, pero sin un LLM: se usa texto del propio documento, así que no se inventa contenido. Resuelve las secciones con el mismo nombre en dos documentos ("Requisitos", "Preguntas frecuentes"), que serían ambiguas por separado.
5. **El tope de 1,200 caracteres** deja preparada la estrategia para documentos futuros con secciones largas: dividiría por filas o párrafos completos y repetiría el encabezado.

### 4. Estructura de los chunks y metadatos
Cada elemento de `chunks` sigue la estructura de la guía (`id`, `text`, `metadata`), y en ChromaDB se guarda así:

| Elemento del registro | Qué contiene en este proyecto | Para qué se utiliza |
|---|---|---|
| `id` | `bca-tc_chunk_02` (documento + número de chunk) | Identificar el fragmento, actualizarlo con `upsert` y citarlo en la respuesta |
| `document` | `text`: prefijo `"{título} · {sección}"` y contenido de la sección | Contexto que recibe el agente |
| `embedding` | Vector de 1536 dimensiones de `text-embedding-3-small` | Comparación semántica con la pregunta; se guarda aparte de los metadatos |
| `metadata` | Valores simples (texto, número o booleano) | Filtrar resultados y mostrar la procedencia |

**Evaluación de los metadatos disponibles** (del índice del Lab 1 y de las propiedades de los PDF):

| Metadato candidato | Origen | Decisión | Uso |
|---|---|---|---|
| `categoria` (Institucional, Tarjeta de Crédito, Crédito Efectivo) | `kb_index.json` | ✅ Se guarda | **Filtro principal** (`where`): el asesor o el agente saben de qué producto se trata |
| `tipo_documento` (Ficha de producto, Información institucional) | `kb_index.json` | ✅ Se guarda | **Filtro** secundario (por ejemplo, solo fichas de producto) |
| `vigente` (bool) | `kb_index.json` | ✅ Se guarda | **Filtro** de control: cuando haya una versión 2.0, se excluyen las fichas no vigentes con `{"vigente": True}` |
| `titulo` | Encabezado H1 del Markdown (coincide con `/Title` del PDF) | ✅ Se guarda | **Contexto** y cita |
| `fuente` (nombre del PDF) | `kb_index.json` | ✅ Se guarda | **Contexto** y cita: el asesor puede abrir el PDF original |
| `seccion` | Encabezado de la sección | ✅ Se guarda | **Contexto**: indica en qué parte del documento está el dato |
| `paginas` (`"1,2"`) | Calculado de los `PageBreak` | ✅ Se guarda como **texto** (Chroma no admite listas) | **Contexto** y cita |
| `version_documento`, `fecha_publicacion` | Subtítulo del PDF / `kb_index.json` | ✅ Se guardan | **Contexto**: el asesor sabe qué versión de la condición está citando |
| `numero_chunk` | Calculado | ✅ Se guarda | **Trazabilidad** y orden dentro del documento |
| `origen` ("sintético…") | `kb_index.json` | ✅ Se guarda | **Trazabilidad**: evidencia de que no hay datos reales |
| `/Author` del PDF | Propiedades del PDF | ❌ Se descarta | Es "BancAnalytics" en todos: no distingue ni filtra nada |
| `/Creator`, `/Subject`, `/Keywords` | Propiedades del PDF | ❌ Se descartan | Vienen vacíos o `(unspecified)` |
| `/Producer`, `/CreationDate` del PDF | Propiedades del PDF | ❌ Se descartan | Describen cómo se generó el archivo (ReportLab), no el contenido; la fecha útil es `fecha_publicacion` |
| `paginas` como lista, `id` del Lab 1 | `kb_index.json` | ❌ No se usan tal cual | Chroma solo admite valores simples; el chunking del Lab 3 genera sus propios ids |

In [9]:
# Estructura final de un chunk (como la referencia de la guía)
print(json.dumps(chunks[8], ensure_ascii=False, indent=2))

# Todos los metadatos son valores simples (str, int, float o bool)
assert all(isinstance(v, (str, int, float, bool)) for c in chunks for v in c["metadata"].values())
print("\nCategorías disponibles para filtrar:", sorted({c["metadata"]["categoria"] for c in chunks}))

{
  "id": "bca-tc_chunk_02",
  "text": "Tarjeta de Crédito BancAnalytics · Variantes de plástico\n\n- Ingreso mínimo mensual → Clásica: S/ 1,200 | Oro: S/ 3,000 | Platinum: S/ 6,000 | Black: S/ 15,000\n- Línea de crédito → Clásica: S/ 500 -6,000 | Oro: S/ 3,000 - 15,000 | Platinum: S/ 10,000 - 40,000 | Black: Desde S/ 40,000\n- Membresía anual → Clásica: S/ 0 | Oro: S/ 120 (exonerable) | Platinum: S/ 240 (exonerable) | Black: S/ 600 (exonerable)\n- TEA compras (ref.) → Clásica: 79.9% | Oro: 69.9% | Platinum: 54.9% | Black: 39.9%\n- Puntos por cada S/ 1 → Clásica: 1 pto. x S/ 5 | Oro: 1 pto. x S/ 3 | Platinum: 1 pto. x S/ 2 | Black: 1 pto. x S/ 1\n- Cashback → Clásica: — | Oro: 1% en supermercados | Platinum: 2% supermercados y grifos | Black: 3% en restaurantes y viajes\n- Salas VIP aeropuerto → Clásica: — | Oro: — | Platinum: 4 accesos/año | Black: Ilimitado + 1 invitado\n- Seguro de viaje → Clásica: — | Oro: — | Platinum: Incluido | Black: Incluido (cobertura ampliada)\n- Otros benef

#### Pregunta 2 · ¿Qué metadatos seleccionaron? ¿Cuáles usarán para filtrar y cuáles para enriquecer el contexto de las respuestas?

- **Seleccionados:** `fuente`, `titulo`, `seccion`, `categoria`, `tipo_documento`, `numero_chunk`, `paginas`, `version_documento`, `fecha_publicacion`, `vigente` y `origen`. Incluyen los 4 mínimos que pide la guía (`fuente`, `titulo`, `categoria`, `numero_chunk`).
- **Para filtrar** (`where` de Chroma):
  - **`categoria`** es el filtro principal. El copiloto atiende tres temas, y cuando el asesor ya está en una atención de Crédito Efectivo, limitar la búsqueda evita traer los "Requisitos" o las "Preguntas frecuentes" de la tarjeta, que son secciones con nombres y vocabulario muy parecidos (sección 6).
  - `tipo_documento` y `vigente` quedan disponibles para cuando haya más tipos de documentos y versiones.
- **Para enriquecer el contexto y citar:** `titulo`, `fuente`, `seccion`, `paginas`, `version_documento` e `id`. Se insertan en el bloque de contexto como `[id | fuente | sección | págs.]` para que el agente cite la procedencia en la sección *5) Fuentes* de su respuesta, igual que en el Laboratorio 2.
- **Solo trazabilidad:** `numero_chunk` y `origen`.
- **Descartados:** las propiedades del PDF que están vacías o no distinguen documentos (autor, creator, keywords, producer, fecha de creación del archivo).

### 5. Embeddings con Azure OpenAI (`text-embedding-3-small`)
**Paso 1.** Se instalaron `agent-framework-openai`, `chromadb`, `azure-identity` y `python-dotenv` (agregados a `requirements.txt`), y se añadió `FOUNDRY_EMBEDDING_DEPLOYMENT="text-embedding-3-small"` al `.env`. Se usa el **mismo endpoint** que el LLM, porque ambos deployments están en el mismo recurso de Foundry, y la autenticación es con la identidad asignada (sin API keys).

**Paso 2.** Se prueba la generación de un vector con `OpenAIEmbeddingClient` de Microsoft Agent Framework. A diferencia de la teoría, no se usa Sentence Transformers. El **mismo cliente y modelo** vectorizan los chunks (sección 6) y las preguntas (`recuperar_fragmentos`).

In [10]:
cliente_embeddings = OpenAIEmbeddingClient(
    model=modelo_embedding,  # text-embedding-3-small
    base_url=endpoint_llm,   # mismo endpoint del Laboratorio 2
    credential=credential,
)

prueba = await cliente_embeddings.get_embeddings(
    ["Consulta ficticia de prueba para el Proyecto Integrador."]
)
print("Dimensiones del vector:", len(prueba[0].vector))
print("Primeros valores:", [round(v, 4) for v in prueba[0].vector[:5]])

Dimensiones del vector: 1536
Primeros valores: [-0.0025, -0.0073, -0.0003, 0.0038, -0.0242]


### 6. Almacenamiento y recuperación en ChromaDB
#### 6.A Guardar chunks, vectores y metadatos
Se usa una base **local y persistente** en `./chroma_proyecto`, que está en `.gitignore` y no se sube al repositorio. El nombre de la colección incluye la estrategia y su versión (`bancanalytics_estructura_v1`): si cambia el chunking o el modelo de embeddings, se crea una colección nueva y no se mezclan configuraciones. Además, antes del `upsert` se **eliminan los ids que ya no existen** en la lista `chunks`, porque Chroma no los borra automáticamente.

In [11]:
import chromadb
from chromadb.config import Settings

if not chunks:
    raise ValueError("Primero deben generar los chunks del proyecto.")

# Los embeddings se calculan en Azure mediante Agent Framework
textos = [chunk["text"] for chunk in chunks]
resultado_embeddings = await cliente_embeddings.get_embeddings(textos)
vectores = [item.vector for item in resultado_embeddings]

chroma = chromadb.PersistentClient(path="./chroma_proyecto", settings=Settings(anonymized_telemetry=False))
coleccion = chroma.get_or_create_collection(
    name="bancanalytics_estructura_v1",
    embedding_function=None,                      # Chroma no genera embeddings propios
    configuration={"hnsw": {"space": "cosine"}},  # distancia = 1 - similitud del coseno
    metadata={"modelo_embedding": modelo_embedding, "chunking": "estructura-md-max1200-v1"},
)

# Registros antiguos: se borran los ids que ya no están en la lista de chunks
ids_actuales = {chunk["id"] for chunk in chunks}
huerfanos = [i for i in coleccion.get(include=[])["ids"] if i not in ids_actuales]
if huerfanos:
    coleccion.delete(ids=huerfanos)
print("Registros antiguos eliminados:", len(huerfanos))

coleccion.upsert(
    ids=[chunk["id"] for chunk in chunks],
    documents=textos,
    metadatas=[chunk["metadata"] for chunk in chunks],
    embeddings=vectores,
)
print("Chunks almacenados:", coleccion.count())
print("Colecciones en la base local:", [c.name for c in chroma.list_collections()])

Registros antiguos eliminados: 0
Chunks almacenados: 20
Colecciones en la base local: ['bancanalytics_estructura_v1']


#### 6.B Recuperar información y aplicar un filtro
`recuperar_fragmentos` es la función de referencia de la guía: vectoriza la pregunta con el **mismo** modelo y permite filtrar por `categoria`.

In [12]:
async def recuperar_fragmentos(pregunta: str, categoria: str | None = None, top_k: int = 3):
    if coleccion.count() == 0:
        return []

    embedding = await cliente_embeddings.get_embeddings([pregunta])
    parametros = {
        "query_embeddings": [embedding[0].vector],
        "n_results": min(top_k, coleccion.count()),
        "include": ["documents", "metadatas", "distances"],
    }
    if categoria is not None:
        parametros["where"] = {"categoria": categoria}

    resultados = coleccion.query(**parametros)
    return [
        {"id": id_chunk, "text": texto, "metadata": metadatos, "distancia": distancia}
        for id_chunk, texto, metadatos, distancia in zip(
            resultados["ids"][0],
            resultados["documents"][0],
            resultados["metadatas"][0],
            resultados["distances"][0],
        )
    ]

def tabla_fragmentos(fragmentos):
    return pd.DataFrame([{
        "id": f["id"], "categoria": f["metadata"]["categoria"], "seccion": f["metadata"]["seccion"],
        "fuente": f["metadata"]["fuente"], "distancia": round(f["distancia"], 4),
    } for f in fragmentos])

In [13]:
# Pregunta ambigua: "requisitos" existe en la Tarjeta y en el Crédito Efectivo
pregunta = "¿Qué requisitos debe cumplir el cliente y qué ingreso mínimo necesita?"

sin_filtro = await recuperar_fragmentos(pregunta, top_k=4)
con_filtro = await recuperar_fragmentos(pregunta, categoria="Crédito Efectivo", top_k=4)

print("PREGUNTA:", pregunta)
print("\nSIN FILTRO")
display(tabla_fragmentos(sin_filtro))
print("CON FILTRO categoria='Crédito Efectivo'")
display(tabla_fragmentos(con_filtro))

for item in con_filtro[:1]:
    print(item["id"], item["metadata"]["fuente"], round(item["distancia"], 4))
    print(item["text"], "\n")

PREGUNTA: ¿Qué requisitos debe cumplir el cliente y qué ingreso mínimo necesita?

SIN FILTRO


,id,categoria,seccion,fuente,distancia
0,bca-ce_chunk_05,Crédito Efectivo,Requisitos,03_BancAnalytics_Credito_Efectivo.pdf,0.3960
1,bca-tc_chunk_04,Tarjeta de Crédito,Requisitos,02_BancAnalytics_Tarjeta_de_Credito.pdf,0.4878
2,bca-ce_chunk_06,Crédito Efectivo,Proceso de venta con el copiloto,03_BancAnalytics_Credito_Efectivo.pdf,0.5525
3,bca-ce_chunk_03,Crédito Efectivo,Condiciones,03_BancAnalytics_Credito_Efectivo.pdf,0.5721


CON FILTRO categoria='Crédito Efectivo'


,id,categoria,seccion,fuente,distancia
0,bca-ce_chunk_05,Crédito Efectivo,Requisitos,03_BancAnalytics_Credito_Efectivo.pdf,0.3960
1,bca-ce_chunk_06,Crédito Efectivo,Proceso de venta con el copiloto,03_BancAnalytics_Credito_Efectivo.pdf,0.5525
2,bca-ce_chunk_03,Crédito Efectivo,Condiciones,03_BancAnalytics_Credito_Efectivo.pdf,0.5721
3,bca-ce_chunk_02,Crédito Efectivo,Tipos de oferta,03_BancAnalytics_Credito_Efectivo.pdf,0.5955


bca-ce_chunk_05 03_BancAnalytics_Credito_Efectivo.pdf 0.396
Crédito Efectivo BancAnalytics · Requisitos

· Persona natural de 21 a 70 años con DNI vigente (o carné de extranjería).

· Ingreso neto mínimo de S/ 1,000 mensuales.

· Dependiente: 3 meses de antigüedad laboral. Independiente: 12 meses de actividad.

· Calificación Normal en la SBS y capacidad de pago verificada (cuotas totales ≤ 40% del ingreso neto, referencial).

· Con oferta aprobada no se requiere documentación adicional. 



**Comparación sin filtro y con filtro** (pregunta ambigua: *"¿Qué requisitos debe cumplir el cliente y qué ingreso mínimo necesita?"*):

| Posición | Sin filtro | Con `categoria="Crédito Efectivo"` |
|---|---|---|
| 1 | `bca-ce_chunk_05` Requisitos CE (0.396) | `bca-ce_chunk_05` Requisitos CE (0.396) |
| 2 | **`bca-tc_chunk_04` Requisitos Tarjeta (0.488)** | `bca-ce_chunk_06` Proceso de venta (0.553) |
| 3 | `bca-ce_chunk_06` Proceso de venta (0.553) | `bca-ce_chunk_03` Condiciones (0.572) |
| 4 | `bca-ce_chunk_03` Condiciones (0.572) | `bca-ce_chunk_02` Tipos de oferta (0.596) |

- **Sin filtro**, el segundo resultado son los requisitos de la **tarjeta**, que piden otra edad (21 a 75 años) y otro ingreso. Si el asesor está atendiendo un Crédito Efectivo, el agente recibiría dos conjuntos de requisitos contradictorios.
- **Con el filtro**, todos los fragmentos son del documento correcto. **Las distancias de los chunks comunes no cambian** (0.396, 0.553, 0.572): el filtro no reordena ni mejora la similitud, solo **restringe el universo** de búsqueda y deja entrar al siguiente chunk de la misma categoría (Tipos de oferta, 0.596).
- Las distancias son de coseno (1 − similitud): cuanto menor, más parecido. Aun así, el segundo y tercer resultado (≈ 0.55) son temas vecinos, no la respuesta.

### 7. Integración del RAG con el agente del Laboratorio 2
Se mantiene **el mismo agente** del Laboratorio 2: mismo rol, objetivo, restricciones de alcance, formato con la etiqueta `ALCANCE:`, few-shot, memoria (`InMemoryHistoryProvider`) y compactación (`CompactionProvider` + `SummarizationStrategy`, 6 turnos enteros). Hay solo dos cambios en las instrucciones:
1. **`contexto`:** ya no incluye la base de conocimiento completa. Ahora indica que la información de productos llega en cada mensaje como `CONTEXTO RECUPERADO`.
2. **`restricciones`:** se agrega la regla RAG de la guía (usar solo el contexto recuperado, decir que no hay información suficiente si no la sustenta, no inventar condiciones, cifras ni procedimientos, y mencionar la fuente).

Flujo: **pregunta → embedding (`text-embedding-3-small`) → recuperación en ChromaDB → contexto con procedencia → respuesta del agente (`gpt-5.6-luna`)**.

In [14]:
REGLA_RAG = (
    "\n- REGLA RAG: para responder preguntas sobre documentos o condiciones del proyecto, "
    "utiliza únicamente la información del CONTEXTO RECUPERADO. Si el contexto no sustenta "
    "una respuesta, indica que no dispones de información suficiente. No inventes condiciones, "
    "cifras o procedimientos, ni completes con conocimiento general. Menciona la fuente utilizada "
    "con su id y documento. Un fragmento recuperado puede ser parecido a la pregunta y aun así "
    "no contener el dato pedido: verifica que el dato esté escrito en el contexto."
)

datos_rag = {
    **datos,
    "contexto": (
        "Quien te escribe es el ASESOR, no el cliente. País: Perú; regulación: SBS, "
        "Indecopi y Libro de Reclamaciones; moneda: soles (S/). "
        "El asesor puede compartirte el perfil del cliente (ingresos, productos, "
        "ofertas vigentes y necesidad); úsalo durante toda la conversación.\n"
        "Tu única fuente de información de productos y del banco es el CONTEXTO RECUPERADO "
        "que acompaña cada pregunta: fragmentos de documentos sintéticos de BancAnalytics "
        "recuperados de una base vectorial, cada uno con [id | fuente | sección | págs.]."
    ),
    "restricciones": RESTRICCIONES_LAB2 + REGLA_RAG,
    "formato_salida": formato_salida("[bca-tc_chunk_02 | 02_BancAnalytics_Tarjeta_de_Credito.pdf]"),
}
system_prompt_rag = plantilla_prompt.format(**datos_rag)

# Mismo agente del Laboratorio 2 (memoria + compactación con la configuración final), ahora con RAG
agente = crear_agente_compactado(system_prompt_rag, target_count=TURNOS_ENTEROS * 2, threshold=4)

print(f"System prompt Lab 2: {len(system_prompt_lab2):,} caracteres | System prompt Lab 3 (RAG): {len(system_prompt_rag):,} caracteres")

System prompt Lab 2: 17,412 caracteres | System prompt Lab 3 (RAG): 4,715 caracteres


In [15]:
def construir_contexto(fragmentos):
    return "\n\n".join(
        f"[{item['id']} | {item['metadata']['fuente']} | {item['metadata']['seccion']} | "
        f"págs. {item['metadata']['paginas']} | distancia {item['distancia']:.3f}]\n"
        f"Título: {item['metadata']['titulo']}\n"
        f"Contenido: {item['text']}"
        for item in fragmentos
    )

async def preguntar_con_rag(pregunta: str, sesion, categoria: str | None = None,
                            top_k: int = 3, umbral: float | None = None):
    """Adaptación de la función de la guía: usa el agente y la sesión del Laboratorio 2.

    Devuelve (texto de la respuesta, fragmentos recuperados, fragmentos enviados al agente).
    Si se indica un umbral, solo se envían los fragmentos con distancia <= umbral.
    """
    fragmentos = await recuperar_fragmentos(pregunta, categoria=categoria, top_k=top_k)
    enviados = [f for f in fragmentos if umbral is None or f["distancia"] <= umbral]

    if not fragmentos:
        return "No encontré información relacionada en los documentos disponibles.", [], []

    contexto = construir_contexto(enviados) if enviados else "(sin fragmentos suficientemente relacionados)"
    consulta = (
        f"CONTEXTO RECUPERADO:\n{contexto}\n\n"
        f"PREGUNTA:\n{pregunta}\n\n"
        "Responde solo si el contexto sustenta la respuesta. "
        "Si no es suficiente, indícalo explícitamente."
    )
    respuesta = await agente.run(consulta, session=sesion)
    return respuesta.text, fragmentos, enviados

#### 7.1 Conversación de varios turnos con la misma sesión
Se reutiliza **la misma sesión**: el perfil del cliente se da en el turno 1 y los turnos 2 y 3 necesitan **el perfil (memoria) y los documentos (RAG)** al mismo tiempo.

In [16]:
sesion = agente.create_session()

conversacion_rag = [
    "Atiendo a la clienta Rosa P. (DNI ****1234): ingreso neto de S/ 4,500 mensuales, "
    "dependiente con 2 años en su trabajo, tiene tarjeta Clásica y un Crédito Efectivo "
    "pre-aprobado de hasta S/ 20,000 vigente. Quiere consolidar deudas de otros bancos.",
    "Con su ingreso, ¿a qué variante de tarjeta puede hacer upgrade y qué beneficios ganaría?",
    "¿Qué validaciones necesita su oferta antes del desembolso y en cuánto tiempo recibiría el dinero?",
]

for i, mensaje in enumerate(conversacion_rag, start=1):
    texto, fragmentos, _ = await preguntar_con_rag(mensaje, sesion)
    print("=" * 100)
    print(f"[Turno {i}] ASESOR: {mensaje}")
    print("Fragmentos:", [(f["id"], round(f["distancia"], 3)) for f in fragmentos])
    print(f"Historial de la sesión: {estado_historial(sesion)}\n")
    print(texto if i > 1 else texto[:500] + " [...]")

[Turno 1] ASESOR: Atiendo a la clienta Rosa P. (DNI ****1234): ingreso neto de S/ 4,500 mensuales, dependiente con 2 años en su trabajo, tiene tarjeta Clásica y un Crédito Efectivo pre-aprobado de hasta S/ 20,000 vigente. Quiere consolidar deudas de otros bancos.
Fragmentos: [('bca-ce_chunk_05', 0.346), ('bca-tc_chunk_04', 0.42), ('bca-tc_chunk_02', 0.443)]
Historial de la sesión: {'almacenados': 2, 'enviados_al_modelo': 2, 'excluidos': 0, 'resumenes': 0}

ALCANCE: EN_ALCANCE

1) **Resumen de la consulta:** Rosa P. percibe S/ 4,500 netos, es dependiente con 2 años de antigüedad, tiene Tarjeta Clásica y una oferta **pre-aprobada** de Crédito Efectivo de hasta S/ 20,000. Desea consolidar deudas de otros bancos.

2) **Respuesta para el asesor:**
- La oferta debe comunicarse como: **“Crédito Efectivo pre-aprobado de hasta S/ 20,000, sujeto a validación”**. No debe presentarse como aprobado. El contexto no indica la fecha exacta de vigencia; debe verif [...]


[Turno 2] ASESOR: Con su ingreso, ¿a qué variante de tarjeta puede hacer upgrade y qué beneficios ganaría?
Fragmentos: [('bca-tc_chunk_05', 0.412), ('bca-tc_chunk_02', 0.459), ('bca-tc_chunk_06', 0.48)]
Historial de la sesión: {'almacenados': 4, 'enviados_al_modelo': 4, 'excluidos': 0, 'resumenes': 0}

ALCANCE: EN_ALCANCE

1) **Resumen de la consulta:** Determinar la variante a la que Rosa podría solicitar un upgrade según su ingreso neto de S/ 4,500 y los beneficios que obtendría.

2) **Respuesta para el asesor:**
- Rosa podría **solicitar un upgrade de Clásica a Oro**, porque su ingreso supera el mínimo de S/ 3,000. El cambio está sujeto a evaluación.
- No alcanzaría el ingreso mínimo informado para Platinum, que es S/ 6,000.
- Con Oro obtendría, frente a la Clásica:
  - **1 punto por cada S/ 3**, en lugar de 1 punto por cada S/ 5.
  - **1% de cashback en supermercados**.
  - **Descuentos en cines y delivery**.
  - Membresía anual de **S/ 120, exonerable** con consumos de **S/ 1,500 

[Turno 3] ASESOR: ¿Qué validaciones necesita su oferta antes del desembolso y en cuánto tiempo recibiría el dinero?
Fragmentos: [('bca-ce_chunk_02', 0.4), ('bca-ce_chunk_06', 0.432), ('bca-ce_chunk_07', 0.469)]
Historial de la sesión: {'almacenados': 6, 'enviados_al_modelo': 6, 'excluidos': 0, 'resumenes': 0}

ALCANCE: EN_ALCANCE

1) **Resumen de la consulta:** Validaciones y plazo de desembolso de la oferta pre-aprobada de Crédito Efectivo de Rosa.

2) **Respuesta para el asesor:**
- La oferta de Rosa es **pre-aprobada de hasta S/ 20,000, sujeta a validación**; no debe comunicarse como aprobada.
- Antes del desembolso se debe validar:
  - Su identidad.
  - Sus ingresos actualizados.
  - Que su situación crediticia no haya cambiado.
- Como resultado de la validación, el monto podría reducirse o ajustarse y también podría cambiar la tasa.
- Una vez completada la validación, el desembolso puede realizarse **hasta en 24 horas**. El contexto no precisa si el abono será en una cuenta especí

**Observaciones de la integración:**
- **El system prompt bajó de 17,412 a 4,715 caracteres.** La base de conocimiento ya no viaja completa en cada turno: ahora se envían solo los 3 fragmentos recuperados (como máximo unos 3.7 mil caracteres, porque cada chunk mide menos de 1,250), con su procedencia `[id | fuente | sección | págs. | distancia]`.
- **Memoria y RAG juntos:** en el **turno 2** (*"Con su ingreso, ¿a qué variante…?"*) la pregunta no repite el ingreso. El agente tomó los **S/ 4,500 del turno 1** (memoria) y el ingreso mínimo de cada variante del chunk `bca-tc_chunk_02` (RAG). Recomendó **Oro**, descartó Platinum (S/ 6,000) y citó las fuentes.
- En el **turno 3** respondió las validaciones y el plazo de desembolso de una oferta pre-aprobada (identidad, ingresos actualizados, situación crediticia; hasta 24 horas), a partir de `bca-ce_chunk_02`. Mantuvo la **regla clave**: *"pre-aprobada de hasta S/ 20,000, sujeta a validación"*.
- **Respeta la regla RAG:** en el turno 1 respondió que *"el contexto no indica la fecha exacta de vigencia"* en lugar de suponerla.
- **Detalle de diseño a mejorar:** la función de la guía busca **en cada mensaje**, incluso en el turno 1, que solo describe el perfil. Allí recuperó requisitos y variantes que nadie pidió. Además, el mensaje que queda en el historial incluye el contexto recuperado, así que el historial crece más rápido que en el Lab 2. La compactación lo controla (el prompt de resumen pide no copiar los fragmentos), pero es un argumento para convertir la búsqueda en una Tool (sección 9).

### 8. Pruebas y validación del RAG
Cada prueba usa una **sesión nueva** (es independiente) y registra: pregunta, filtro, chunks y fuentes recuperados con su distancia, respuesta y evaluación.

La **evaluación automática** combina dos comprobaciones:
- **Recuperación:** ¿el chunk que contiene el dato está entre los recuperados?
- **Respuesta:** ¿la respuesta contiene el dato esperado (y cita la fuente)? En la prueba 5, ¿reconoce que no tiene información y **no** inventa una cifra?

Resultado: `correcto` si se cumplen ambas, `parcial` si se cumple una e `incorrecto` si ninguna. La justificación final se revisa manualmente debajo de la tabla.

In [17]:
def normalizar(t):
    return re.sub(r"\s+", " ", (t or "").lower())

FRASES_SIN_INFO = ["no dispongo", "no tengo información", "no cuento con", "no se indica", "no indica",
                   "no especifica", "no se especifica", "no figura", "no aparece", "no incluye",
                   "información suficiente", "no está", "no hay información", "no se detalla", "no detalla"]

PRUEBAS = [
    {"n": 1, "tipo": "Pregunta directa",
     "pregunta": "¿Qué ingreso mínimo mensual pide la tarjeta Platinum?",
     "categoria": None, "chunk_esperado": "bca-tc_chunk_02", "debe_contener": ["6,000"]},
    {"n": 2, "tipo": "Pregunta reformulada",
     "pregunta": "¿Cuánto tiene que ganar al mes una persona para que le den la Platinum?",
     "categoria": None, "chunk_esperado": "bca-tc_chunk_02", "debe_contener": ["6,000"]},
    {"n": 3, "tipo": "Información en otro chunk",
     "pregunta": "Un cliente quiere poner un reclamo: ¿dónde lo registra y en qué plazo le responden?",
     "categoria": None, "chunk_esperado": "bca-inst_chunk_05", "debe_contener": ["15 días"]},
    {"n": 4, "tipo": "Pregunta con filtro",
     "pregunta": "¿Qué requisitos debe cumplir el cliente y qué ingreso mínimo necesita?",
     "categoria": "Crédito Efectivo", "chunk_esperado": "bca-ce_chunk_05", "debe_contener": ["1,000", "70"]},
    {"n": 5, "tipo": "Información ausente",
     "pregunta": "¿Cuál es la suma asegurada en dólares del seguro de viaje de la tarjeta Black?",
     "categoria": None, "chunk_esperado": None, "debe_contener": []},
]

async def ejecutar_prueba(prueba, umbral=None):
    sesion_prueba = agente.create_session()  # prueba independiente
    texto, fragmentos, enviados = await preguntar_con_rag(
        prueba["pregunta"], sesion_prueba, categoria=prueba["categoria"], umbral=umbral)
    ids = [f["id"] for f in fragmentos]
    r = normalizar(texto)
    if prueba["chunk_esperado"]:
        ok_recuperacion = prueba["chunk_esperado"] in ids
        ok_respuesta = all(normalizar(x) in r for x in prueba["debe_contener"]) and "fuente" in r
    else:  # información ausente: no debe inventar una cifra en dólares
        ok_recuperacion = True  # se recupera "lo más parecido"; no es evidencia (ver análisis)
        ok_respuesta = any(f in r for f in FRASES_SIN_INFO) and not re.search(r"(us\$|usd|\$ ?\d)", r)
    evaluacion = "correcto" if ok_recuperacion and ok_respuesta else ("parcial" if ok_recuperacion or ok_respuesta else "incorrecto")
    return {
        "prueba": f"{prueba['n']}. {prueba['tipo']}",
        "pregunta": prueba["pregunta"],
        "filtro": prueba["categoria"] or "—",
        "chunks (distancia)": ", ".join(f"{f['id']} ({f['distancia']:.3f})" for f in fragmentos),
        "fuentes": ", ".join(sorted({f["metadata"]["fuente"] for f in fragmentos})),
        "recuperacion_ok": ok_recuperacion,
        "respuesta_ok": ok_respuesta,
        "evaluacion": evaluacion,
        "respuesta": texto,
        "_fragmentos": fragmentos,
    }

resultados_pruebas = [await ejecutar_prueba(p) for p in PRUEBAS]
df_pruebas = pd.DataFrame(resultados_pruebas).drop(columns=["respuesta", "_fragmentos"])
df_pruebas

,prueba,pregunta,filtro,chunks (distancia),fuentes,recuperacion_ok,respuesta_ok,evaluacion
0,1. Pregunta directa,¿Qué ingreso mínimo mensual pide la tarjeta Platinum?,—,"bca-tc_chunk_02 (0.309), bca-tc_chunk_05 (0.397), bca-tc_chunk_04 (0.507)",02_BancAnalytics_Tarjeta_de_Credito.pdf,True,True,correcto
1,2. Pregunta reformulada,¿Cuánto tiene que ganar al mes una persona para que le den la Platinum?,—,"bca-tc_chunk_02 (0.423), bca-tc_chunk_05 (0.540), bca-ce_chunk_05 (0.584)","02_BancAnalytics_Tarjeta_de_Credito.pdf, 03_BancAnalytics_Credito_Efectivo.pdf",True,True,correcto
2,3. Información en otro chunk,Un cliente quiere poner un reclamo: ¿dónde lo registra y en qué plazo le responden?,—,"bca-inst_chunk_05 (0.447), bca-ce_chunk_07 (0.601), bca-ce_chunk_06 (0.605)","01_BancAnalytics_Informacion_Institucional.pdf, 03_BancAnalytics_Credito_Efectivo.pdf",True,True,correcto
3,4. Pregunta con filtro,¿Qué requisitos debe cumplir el cliente y qué ingreso mínimo necesita?,Crédito Efectivo,"bca-ce_chunk_05 (0.396), bca-ce_chunk_06 (0.553), bca-ce_chunk_03 (0.572)",03_BancAnalytics_Credito_Efectivo.pdf,True,True,correcto
4,5. Información ausente,¿Cuál es la suma asegurada en dólares del seguro de viaje de la tarjeta Black?,—,"bca-tc_chunk_02 (0.464), bca-tc_chunk_05 (0.543), bca-tc_chunk_03 (0.560)",02_BancAnalytics_Tarjeta_de_Credito.pdf,True,True,correcto


In [18]:
# Respuestas completas (evidencia)
for res in resultados_pruebas:
    print("=" * 100)
    print(f"{res['prueba']} | filtro: {res['filtro']} | evaluación: {res['evaluacion'].upper()}")
    print(f"PREGUNTA: {res['pregunta']}")
    print(f"CHUNKS: {res['chunks (distancia)']}\n")
    print(res["respuesta"])

1. Pregunta directa | filtro: — | evaluación: CORRECTO
PREGUNTA: ¿Qué ingreso mínimo mensual pide la tarjeta Platinum?
CHUNKS: bca-tc_chunk_02 (0.309), bca-tc_chunk_05 (0.397), bca-tc_chunk_04 (0.507)

ALCANCE: EN_ALCANCE

1) **Resumen de la consulta:** Ingreso mínimo mensual requerido para la Tarjeta Platinum.

2) **Respuesta para el asesor:** La Tarjeta Platinum requiere un ingreso mínimo mensual de **S/ 6,000**.

3) **Mensaje sugerido para el cliente:** “La Tarjeta de Crédito BancAnalytics Platinum solicita un ingreso mínimo mensual de S/ 6,000, sujeto a evaluación crediticia y demás requisitos aplicables.”

4) **Oportunidades de oferta:** Si el cliente cumple el ingreso mínimo y cuenta con una oferta vigente en su perfil, prioriza la variante y línea indicadas en dicha oferta.

5) **Fuentes:** [bca-tc_chunk_02 | 02_BancAnalytics_Tarjeta_de_Credito.pdf]; [bca-tc_chunk_04 | 02_BancAnalytics_Tarjeta_de_Credito.pdf]
2. Pregunta reformulada | filtro: — | evaluación: CORRECTO
PREGUNTA: ¿

In [19]:
# Prueba 4: ¿qué cambia el filtro? Misma pregunta sin filtro
sin_filtro_p4 = await recuperar_fragmentos(PRUEBAS[3]["pregunta"])
print("Prueba 4 SIN filtro:")
display(tabla_fragmentos(sin_filtro_p4))
print("Prueba 4 CON filtro 'Crédito Efectivo':")
display(tabla_fragmentos(resultados_pruebas[3]["_fragmentos"]))

Prueba 4 SIN filtro:


,id,categoria,seccion,fuente,distancia
0,bca-ce_chunk_05,Crédito Efectivo,Requisitos,03_BancAnalytics_Credito_Efectivo.pdf,0.3960
1,bca-tc_chunk_04,Tarjeta de Crédito,Requisitos,02_BancAnalytics_Tarjeta_de_Credito.pdf,0.4878
2,bca-ce_chunk_06,Crédito Efectivo,Proceso de venta con el copiloto,03_BancAnalytics_Credito_Efectivo.pdf,0.5525


Prueba 4 CON filtro 'Crédito Efectivo':


,id,categoria,seccion,fuente,distancia
0,bca-ce_chunk_05,Crédito Efectivo,Requisitos,03_BancAnalytics_Credito_Efectivo.pdf,0.3960
1,bca-ce_chunk_06,Crédito Efectivo,Proceso de venta con el copiloto,03_BancAnalytics_Credito_Efectivo.pdf,0.5525
2,bca-ce_chunk_03,Crédito Efectivo,Condiciones,03_BancAnalytics_Credito_Efectivo.pdf,0.5721


#### 8.1 ¿Sirve un umbral de distancia para la prueba 5?
La guía advierte que la búsqueda vectorial siempre devuelve "lo más parecido", aunque no contenga la respuesta. Antes de usar un umbral, se miden las distancias del **mejor chunk** en preguntas con respuesta y en preguntas sin respuesta en los documentos.

In [20]:
preguntas_umbral = [
    # (pregunta, ¿la respuesta está en los documentos?)
    ("¿Qué ingreso mínimo mensual pide la tarjeta Platinum?", True),
    ("¿Cuánto tiene que ganar al mes una persona para que le den la Platinum?", True),
    ("Un cliente quiere poner un reclamo: ¿dónde lo registra y en qué plazo le responden?", True),
    ("¿Puede el cliente adelantar pagos del Crédito Efectivo sin penalidad?", True),
    ("¿Cuánto cobran por disponer efectivo en cajeros de otros bancos?", True),
    ("¿Los Puntos Analytics vencen?", True),
    ("¿Cuál es la suma asegurada en dólares del seguro de viaje de la tarjeta Black?", False),
    ("¿Cuál es el límite de retiro diario en cajeros con la tarjeta Oro?", False),
    ("¿Qué tasa de interés paga una cuenta de ahorros de BancAnalytics?", False),
    ("¿Cuántos días de vacaciones tienen los asesores del banco?", False),
]
filas = []
for p, tiene_respuesta in preguntas_umbral:
    frags = await recuperar_fragmentos(p, top_k=1)
    filas.append({"pregunta": p, "respuesta_en_documentos": tiene_respuesta,
                  "mejor_chunk": frags[0]["id"], "distancia": round(frags[0]["distancia"], 4)})
df_umbral = pd.DataFrame(filas).sort_values("distancia")
display(df_umbral)

con = df_umbral[df_umbral.respuesta_en_documentos].distancia
sin = df_umbral[~df_umbral.respuesta_en_documentos].distancia
print(f"Con respuesta: {con.min():.3f} a {con.max():.3f} | Sin respuesta: {sin.min():.3f} a {sin.max():.3f}")
print("¿Se separan con un umbral?", "Sí" if con.max() < sin.min() else "No: los rangos se superponen")

,pregunta,respuesta_en_documentos,mejor_chunk,distancia
0,¿Qué ingreso mínimo mensual pide la tarjeta Platinum?,True,bca-tc_chunk_02,0.3094
3,¿Puede el cliente adelantar pagos del Crédito Efectivo sin penalidad?,True,bca-ce_chunk_07,0.3460
8,¿Qué tasa de interés paga una cuenta de ahorros de BancAnalytics?,False,bca-ce_chunk_03,0.3572
4,¿Cuánto cobran por disponer efectivo en cajeros de otros bancos?,True,bca-tc_chunk_03,0.4011
1,¿Cuánto tiene que ganar al mes una persona para que le den la Platinum?,True,bca-tc_chunk_02,0.4230
2,Un cliente quiere poner un reclamo: ¿dónde lo registra y en qué plazo le responden?,True,bca-inst_chunk_05,0.4467
6,¿Cuál es la suma asegurada en dólares del seguro de viaje de la tarjeta Black?,False,bca-tc_chunk_02,0.4640
5,¿Los Puntos Analytics vencen?,True,bca-tc_chunk_06,0.4755
7,¿Cuál es el límite de retiro diario en cajeros con la tarjeta Oro?,False,bca-tc_chunk_02,0.5199
9,¿Cuántos días de vacaciones tienen los asesores del banco?,False,bca-inst_chunk_03,0.5680


Con respuesta: 0.309 a 0.475 | Sin respuesta: 0.357 a 0.568
¿Se separan con un umbral? No: los rangos se superponen


**Resultado de las pruebas** (evaluación automática revisada manualmente):

| Prueba | Pregunta | Filtro | Chunks recuperados (distancia) | Respuesta | Evaluación | Justificación |
|---|---|---|---|---|---|---|
| 1. Directa | ¿Qué ingreso mínimo mensual pide la tarjeta Platinum? | — | **tc_02 (0.309)**, tc_05 (0.397), tc_04 (0.507) | S/ 6,000; cita `bca-tc_chunk_02` | ✅ Correcto | El dato está literal en la tabla de variantes, que fue el primer resultado. Detalle menor: también cita `bca-tc_chunk_04`, que no se usó |
| 2. Reformulada | ¿Cuánto tiene que ganar al mes una persona para que le den la Platinum? | — | **tc_02 (0.423)**, tc_05 (0.540), ce_05 (0.584) | S/ 6,000; ofrece Oro o Clásica si no alcanza | ✅ Correcto | Sin las palabras "ingreso mínimo", el embedding recuperó el mismo chunk. La distancia sube de 0.309 a 0.423, pero sigue primero |
| 3. Otro chunk | Un cliente quiere poner un reclamo: ¿dónde lo registra y en qué plazo le responden? | — | **inst_05 (0.447)**, ce_07 (0.601), ce_06 (0.605) | Libro de Reclamaciones virtual o físico; hasta 15 días hábiles | ✅ Correcto | Recuperó otra sección y otro documento (Institucional). Los otros 2 chunks eran irrelevantes (> 0.60) y el agente no los usó |
| 4. Con filtro | ¿Qué requisitos debe cumplir el cliente y qué ingreso mínimo necesita? | `Crédito Efectivo` | **ce_05 (0.396)**, ce_06 (0.553), ce_03 (0.572) | 21 a 70 años, S/ 1,000, antigüedad, SBS Normal, 40 % | ✅ Correcto | Sin filtro, el 2.º resultado eran los requisitos de la **tarjeta** (tc_04, 0.488). El filtro los excluyó y la respuesta no mezcla edades ni ingresos |
| 5. Ausente | ¿Cuál es la suma asegurada en dólares del seguro de viaje de la tarjeta Black? | — | tc_02 (0.464), tc_05 (0.543), tc_03 (0.560) | *"no especifica la suma asegurada en dólares. No dispongo de información suficiente"* | ✅ Correcto | Caso difícil a propósito: el chunk recuperado **sí** menciona el seguro de viaje de la Black ("Incluido (cobertura ampliada)"), pero no su monto. El agente dijo lo que existe y lo que falta, y no inventó una cifra |

**¿Sirve un umbral de distancia? No, con estos datos.** Los rangos se superponen:
- Preguntas con respuesta: 0.309 a 0.475.
- Preguntas sin respuesta: 0.357 a 0.568.

Por ejemplo, *"¿Qué tasa paga una cuenta de ahorros?"* (fuera de la especialidad, sin respuesta) obtuvo **0.357**, más cerca que la pregunta de los Puntos Analytics (0.475), que sí tiene respuesta. La suma asegurada (0.464) también queda dentro del rango de las respuestas válidas. Un umbral bajo (≈ 0.35) descartaría respuestas buenas, y uno alto (≈ 0.48) no filtraría nada. **Decisión:** no se usa umbral (`umbral=None`). El control de la prueba 5 descansa en la **regla RAG del prompt**, que exige verificar que el dato esté escrito en el contexto, y en la revisión del contenido, no en la distancia. El parámetro `umbral` queda en `preguntar_con_rag` para revisarlo cuando haya más documentos y un conjunto de evaluación más grande.

#### Pregunta 3 · ¿En qué preguntas funcionó bien el RAG y qué dificultades observaron en la recuperación o en las respuestas?

**Funcionó bien:**
- **Preguntas de dato puntual en una tabla** (pruebas 1 y 2): la conversión de tablas a filas `Concepto → Columna: valor` hizo que el chunk correcto quedara siempre primero, incluso con una pregunta reformulada.
- **Preguntas de otra sección o de otro documento** (prueba 3) y **preguntas con filtro** (prueba 4): el metadato `categoria` eliminó los requisitos del otro producto.
- **Conversación con memoria** (sección 7.1): el agente combinó el perfil de la sesión con los fragmentos recuperados, mantuvo la regla de la oferta pre-aprobada y citó las fuentes.
- **Información ausente** (prueba 5): reconoció qué parte del dato no existía aunque el fragmento recuperado fuera muy parecido.

**Dificultades observadas:**
1. **Similitud ≠ evidencia.** En la prueba 5 el chunk recuperado tenía una distancia (0.464) mejor que la de preguntas con respuesta. Solo la instrucción del prompt evitó inventar el monto. El umbral de distancia no sirvió para separar los casos (8.1).
2. **Ruido en los últimos resultados.** Con `top_k=3`, el 2.º y 3.º fragmento suelen ser temas vecinos (distancias de 0.55 a 0.60). El agente no los usó, pero a veces los citó en *Fuentes* (prueba 1). Un **reranking** (`chunking.ipynb`) o un `top_k` dinámico reduciría el ruido.
3. **Secciones con el mismo nombre en documentos distintos.** Sin filtro, los "Requisitos" de la tarjeta compiten con los del Crédito Efectivo (0.488 frente a 0.396). Hoy el filtro lo elige quien llama a la función. Lo ideal es que lo infiera el agente a partir del contexto de la atención (Tool con parámetro `categoria`).
4. **Recuperación en cada turno.** Mensajes que no son preguntas (el perfil del turno 1) también disparan una búsqueda, y el contexto recuperado queda guardado en el historial de la sesión.
5. **Calidad del documento de origen.** Errores de extracción como *"consumos de / 1,500(Or)"* pasan tal cual al contexto. El RAG no corrige la fuente.
6. **Evaluación por palabras clave.** Sirve como primer filtro (las 5 pruebas pasaron), pero no detecta matices, como una fuente citada de más. Se requiere la revisión manual de la tabla anterior y, más adelante, un LLM-as-a-judge (clases 07 y 09).

### 9. Diseño de Tools y Agent as a Tool (solo planificación)
> En este laboratorio **no se implementan** Tools, MCP ni Agent as a Tool. Esta sección solo documenta el diseño para las próximas etapas, siguiendo `tools.ipynb` (*Componentes de una Tool*, *Tools vs. RAG*), `agent-tool.ipynb` y `mcp.ipynb`.

#### 9.A Tools propuestas
| Nombre | Propósito (cuándo debe invocarse) | Entradas | Salidas | Fuente | Justificación (por qué no basta el LLM) |
|---|---|---|---|---|---|
| `buscar_conocimiento` | Recuperar condiciones de producto, canales o reclamos. El agente la invoca **solo cuando la pregunta lo requiere** (no en el saludo ni en los cálculos) y puede reintentar con otra categoría (*Agentic RAG*). | `pregunta: str`, `categoria: Literal["Institucional","Tarjeta de Crédito","Crédito Efectivo"] \| None`, `top_k: int = 3` | Lista de `{id, text, fuente, seccion, paginas, distancia}` | RAG: ChromaDB `bancanalytics_estructura_v1` + `text-embedding-3-small` | Hoy la recuperación se ejecuta **siempre**, antes de llamar al agente, y guarda el contexto en el historial (sección 8). Como tool, el modelo decide cuándo buscar y con qué filtro. El LLM no conoce las condiciones del banco ficticio. |
| `simular_cuota` | Calcular la cuota de un Crédito Efectivo cuando el asesor pide una simulación. | `monto: float` (1,000–150,000), `plazo_meses: int` (6–60), `tea: float`, `incluir_desgravamen: bool = True` | `{cuota_mensual, tasa_mensual, total_intereses, desgravamen_mensual, supuestos}` | Cálculo determinístico (sistema francés: TEM = (1+TEA)^(1/12)−1; desgravamen 0.08 % mensual de la ficha) | Un cálculo financiero no debe depender del modelo (pendiente del Lab 2). Valida los rangos de la ficha y siempre da el mismo resultado para la misma entrada. |
| `obtener_perfil_cliente` | Traer el perfil y las ofertas vigentes al iniciar una atención, en lugar de que el asesor las escriba. | `codigo_cliente: str` (enmascarado) | `{ingreso_neto, situacion_laboral, productos, ofertas: [{producto, tipo (pre-aprobada/aprobada), monto, tea, vence}]}` | Base de datos sintética (SQLite, como `localdb/` de la clase) | El LLM no puede ni debe "recordar" datos de clientes. El **tipo de oferta** debe venir de un sistema y no de un texto libre (regla clave pre-aprobada ≠ aprobada). |
| `evaluar_capacidad_pago` | Antes de recomendar un monto o un upgrade: comprobar que las cuotas totales no superen el 40 % del ingreso neto. | `ingreso_neto: float`, `cuotas_actuales: float`, `cuota_nueva: float` | `{ratio, cumple: bool, margen_disponible}` | Regla de negocio de la ficha (40 %, referencial) | Es una validación de venta responsable; debe ser exacta y auditable. |
| `recomendar_variante_tarjeta` | Sugerir la variante de tarjeta elegible según el ingreso y el perfil de gasto. | `ingreso_neto: float`, `habitos: list[str]` (viajes, supermercado, etc.) | `{variantes_elegibles, recomendada, motivo}` | Tabla de variantes (ingreso mínimo) + guía del asesor, como datos estructurados | El corte por ingreso mínimo es una regla exacta: no conviene que dependa de que el modelo lea bien una tabla. |
| `moderar_mensaje` | Revisar el mensaje del asesor (entrada) y el mensaje sugerido al cliente (salida). | `texto: str`, `direccion: Literal["entrada","salida"]` | `{accion: PERMITIR/ADVERTIR/BLOQUEAR, motivo, coincidencias}` | Azure AI Content Safety + blocklist `blk-bancanalytics-copiloto` (Lab 1) | Hay riesgos de negocio (código SMS, "aprobado sin validar") que el LLM podría dejar pasar. La política del Lab 1 debe aplicarse siempre. Podría ser middleware en lugar de tool. |
| `registrar_resumen_atencion` | Al cerrar la atención: guardar el resumen y los próximos pasos. | `resumen: str`, `productos_ofrecidos: list[str]`, `proxima_accion: str` | `{id_registro, fecha}` | CRM sintético (archivo o SQLite) | Es una **acción** con efecto fuera del modelo. Requiere confirmación del asesor (aprobación humana, clase 08). |

**MCP (evolución):** cuando estas herramientas las usen otros agentes del banco (hipotecario, ahorro), `buscar_conocimiento`, `obtener_perfil_cliente` y `simular_cuota` pueden publicarse en un **servidor MCP**. Así se definen una sola vez, con permisos centralizados, en lugar de copiarlas en cada agente (`mcp.ipynb`, *MCP vs. Tools tradicionales*).

#### 9.B Pregunta 4 · ¿Necesitan un agente especializado que el agente principal pueda utilizar como herramienta?

**Por ahora no: un único agente es suficiente.**
- **El dominio es acotado:** 3 documentos, 20 chunks y tres temas que comparten vocabulario y reglas (ingreso, ofertas, capacidad de pago). La especialización ya está en el system prompt y el alcance se clasifica sin errores (12/12 en el Lab 2).
- **Las capacidades que faltan son cálculos, validaciones o búsquedas**, no razonamiento de otro especialista. Se resuelven mejor con Tools determinísticas (9.A). Un sub-agente agregaría una llamada al LLM más, con más latencia y costo, lo que es crítico en el call center, donde el cliente espera en línea.
- **Las otras especialidades** (hipotecario, ahorro, CTS, seguros) no son subtareas del copiloto, sino conversaciones distintas. Cuando existan, conviene **derivar** la atención (*handoff* u orquestación, `agent-tool.ipynb` sección 10) en lugar de usar Agent as a Tool, porque el agente principal no debe responder esos temas.

**Candidato a futuro, si las pruebas de calidad lo justifican: `agente_cumplimiento` como herramienta del copiloto.**

| Aspecto | Diseño |
|---|---|
| Responsabilidad | Revisar el **borrador** de respuesta antes de mostrarlo al asesor: que la oferta pre-aprobada no se presente como aprobada, que no haya promesas de desembolso, que no se repitan datos sensibles y que **cada cifra esté sustentada** en los fragmentos citados (Self-RAG / Corrective RAG) |
| Recibe | `borrador: str`, `fragmentos: list[{id, text}]`, `perfil_resumido: str` |
| Devuelve | `{veredicto: APROBADO/CORREGIR, hallazgos: list[str], cifras_sin_sustento: list[str], respuesta_corregida: str \| None}` |
| Cuándo se justifica | Si en la evaluación (clases 07 y 09) aparecen cifras no sustentadas o fallas de la regla clave que el prompt y `moderar_mensaje` no corrigen. Es una tarea de **juicio sobre texto**, que una tool determinística no cubre |

### 10. Resultado esperado del Laboratorio 3

| Criterio de la guía | Evidencia en este notebook |
|---|---|
| ✅ Agente del Laboratorio 2 funcionando, con memoria y compactación | Sección 1: mismo prompt y `CompactionProvider`; al turno 5, 4 mensajes excluidos y 1 resumen, y el dato clave conservado |
| ✅ Al menos dos documentos sintéticos preparados para el RAG | Sección 2: 3 documentos (Institucional, Tarjeta de Crédito, Crédito Efectivo) del Lab 1, sin reprocesar; factura excluida con justificación |
| ✅ Estrategia de chunking elegida, implementada y justificada | Sección 3: división por estructura + contexto, tope de 1,200 caracteres; 20 chunks; pregunta 1 |
| ✅ Estructura de chunks y metadatos, distinguiendo contexto y filtros | Sección 4: `id`, `text`, `metadata` con valores simples; tabla de metadatos evaluados; pregunta 2 |
| ✅ Embeddings con `text-embedding-3-small` mediante Microsoft Agent Framework | Sección 5: `OpenAIEmbeddingClient` con `DefaultAzureCredential`, vector de 1536 dimensiones |
| ✅ Chunks, metadatos y vectores almacenados en ChromaDB | Sección 6.A: colección `bancanalytics_estructura_v1` (cosine, `embedding_function=None`), 20 registros, limpieza de ids huérfanos |
| ✅ Recuperación por similitud y al menos una búsqueda filtrada | Secciones 6.B y 8 (prueba 4): con y sin `where={"categoria": ...}`, con comparación de distancias |
| ✅ Agente respondiendo con `gpt-5.6-luna` a partir del contexto recuperado | Sección 7: `preguntar_con_rag` con el agente y la sesión del Lab 2; conversación de 3 turnos con memoria + RAG |
| ✅ Pruebas documentadas, incluida una pregunta sin respuesta disponible | Sección 8: 5 pruebas (5/5 correctas) con chunks, distancias, respuesta y justificación; análisis de umbral; pregunta 3 |
| ✅ Tabla de Tools propuestas y evaluación de Agent as a Tool | Sección 9: 7 Tools diseñadas (sin implementar) y pregunta 4 |
| ✅ Preguntas de reflexión respondidas dentro del notebook | Preguntas 1 (sección 3), 2 (sección 4), 3 (sección 8) y 4 (sección 9.B) |

**Reflexión:**
- **Lo que mejoró respecto del Laboratorio 2:**
  - El prompt por turno pasó de 17 mil a menos de 5 mil caracteres más el contexto recuperado.
  - Cada respuesta cita el **chunk exacto** (`bca-tc_chunk_02`) y no solo la sección.
  - La base de conocimiento puede crecer sin agrandar el system prompt.
- **Lo que se mantuvo:** el alcance, la regla de la oferta pre-aprobada, la memoria de sesión y la compactación funcionan igual con el RAG.
- **Hallazgo principal:** la distancia vectorial no es un criterio confiable de "hay respuesta". Las pruebas sin respuesta se resolvieron por la instrucción del prompt y no por un umbral.
- **Próximos pasos:**
  1. Convertir la recuperación en la Tool `buscar_conocimiento`, para que el agente decida cuándo buscar y con qué categoría.
  2. Implementar `simular_cuota` y `evaluar_capacidad_pago` como cálculos determinísticos.
  3. Evaluar con un conjunto de preguntas más grande y un LLM-as-a-judge antes de decidir si se necesita el `agente_cumplimiento`.